In [71]:
import pandas as pd
import numpy as np
import os
import re
import datetime
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

# 1. ONLY pull cars that have been successfully deep-scraped!
print("Downloading deep-scraped data from Supabase...")
df = pd.read_sql("SELECT * FROM cars WHERE cylinders IS NOT NULL AND make IS NOT NULL", engine)

# 2. Fill any missing specs with 'unspecified' so OHE doesn't crash
cat_cols = ['condition', 'title_status', 'trim', 'cylinders', 'drive', 'fuel', 'transmission', 'type', 'location']
for col in cat_cols:
    df[col] = df[col].fillna('unspecified')

# 3. Filter outliers
df = df[(df['price'] >= 800) & (df['price'] <= 100000)]
df = df[(df['mileage'] >= 100) & (df['mileage'] <= 300000)]
df = df.dropna(subset=['name', 'price', 'mileage'])

print(f"Rows before outlier removal: {len(df)}")

# 1. Drop extreme classic cars
df = df[df['age'] <= 30]

# 2. Drop unrealistic junk/parts cars (Listed under $1500 are usually scams or non-running)
df = df[df['price'] >= 1500]

# 3. Drop impossible mileage for the age (e.g., a 2-year-old car with 300k miles)
df = df[df['mileage'] <= (df['age'] * 25000)]

print(f"Rows after outlier removal: {len(df)}")

# 4. Extract Year and calculate Age (Still parsing from name for now)
df['year'] = df['name'].astype(str).str.extract(r'(\b(19[0-9]{2}|20[0-2][0-9])\b)')[0]
df = df.dropna(subset=['year'])
df['year'] = df['year'].astype(int)
df['age'] = datetime.datetime.now().year - df['year']

# Clean Location
df['location'] = df['location'].astype(str).str.split('/').str[0].str.lower().str.strip()

print(f"Total clean rows loaded: {len(df)}")
df.head()

Rows before outlier removal: 15164
Rows after outlier removal: 13180
Total clean rows loaded: 13180


,name,url,price,mileage,location,make,year,model,age,predicted_price,difference,condition,title_status,trim,region,cylinders,drive,fuel,transmission,type
0,1998 Toyota Camry Le,https://www.craigslist.org/view/d/everett-1998...,2000,253000.0,everett,toyota,1998,camry,28,3359.88,1359.877197,good,clean,unspecified,seattle,4 cylinders,fwd,gas,automatic,unspecified
1,2007 Gmc Yukon Slt,https://www.craigslist.org/view/d/peoria-2007-...,7000,241000.0,glendale,gmc,2007,yukon,19,4801.31,-2198.689941,good,clean,slt,phoenix,8 cylinders,rwd,gas,automatic,suv
2,2003 Toyota Rav4 Runs Great Ice Cold Ac,https://www.craigslist.org/view/d/margate-2003...,3000,166000.0,margate,toyota,2003,rav4,23,4700.80,1700.798340,good,clean,unspecified,miami,4 cylinders,fwd,gas,automatic,suv
5,2018 Chevrolet Cruze Premier - Fully Loaded * ...,https://www.craigslist.org/view/d/los-angeles-...,6800,129000.0,los angeles,chevy,2018,cruze,8,6926.01,126.012695,excellent,clean,premier,losangeles,4 cylinders,fwd,gas,automatic,sedan
6,2010 Honda Civic 155.000 Original Miles $ 5200,https://www.craigslist.org/view/d/los-angeles-...,5200,155000.0,los angeles,honda,2010,civic,16,4978.28,-221.721680,unspecified,clean,unspecified,losangeles,4 cylinders,unspecified,gas,automatic,unspecified


In [72]:
# Calculate Market Baselines to stabilize AI predictions
print('Calculating market baselines...')
# 1. Group Average Baseline (Exact market average for Year/Make/Model)
df['avg_market_price'] = df.groupby(['year', 'make', 'model'])['price'].transform('mean')
df['avg_make_price'] = df.groupby(['year', 'make'])['price'].transform('mean')
df['avg_market_price'] = df['avg_market_price'].fillna(df['avg_make_price']).fillna(df['price'].mean())

# 2. Estimated MSRP Engine (Back-calculate MSRP using age depreciation)
# Approximating a 10% loss of value per year
df['estimated_msrp'] = df['avg_market_price'] * (1 + 0.10 * df['age'])

# Export lookup table for the Live API scraper to use
lookup = df.groupby(['year', 'make', 'model'])[['avg_market_price']].mean().reset_index()
os.makedirs('../api', exist_ok=True)
lookup.to_csv('../api/avg_prices.csv', index=False)
print('Saved avg_prices.csv lookup table.')

# Define Features (X) and Target (y)
X = df[['age', 'make', 'model', 'trim', 'mileage', 'location', 'condition', 
        'title_status', 'cylinders', 'drive', 'fuel', 'transmission', 'type', 'avg_market_price', 'estimated_msrp']]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# OneHotEncoder
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
cat_features = ['make', 'model', 'trim', 'location', 'condition', 
                'title_status', 'cylinders', 'drive', 'fuel', 'transmission', 'type']

X_train_encoded = ohe.fit_transform(X_train[cat_features])
X_test_encoded = ohe.transform(X_test[cat_features])

X_train_encoded_df = pd.DataFrame(X_train_encoded, columns=ohe.get_feature_names_out(), index=X_train.index)
X_test_encoded_df = pd.DataFrame(X_test_encoded, columns=ohe.get_feature_names_out(), index=X_test.index)

X_train_num = X_train[['age', 'mileage', 'avg_market_price', 'estimated_msrp']]
X_test_num = X_test[['age', 'mileage', 'avg_market_price', 'estimated_msrp']]

X_train_final = pd.concat([X_train_num, X_train_encoded_df], axis=1)
X_test_final = pd.concat([X_test_num, X_test_encoded_df], axis=1)

print(f"Training matrix shape: {X_train_final.shape}")

Calculating market baselines...
Saved avg_prices.csv lookup table.
Training matrix shape: (10544, 2498)


In [73]:
# Train XGBoost
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

print("Training model on data...")
model.fit(X_train_final, y_train)
print("Training complete!")

Training model on data...
Training complete!


In [74]:
# Predict
predictions = model.predict(X_test_final)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("Model Performance:")
print(f"Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"R-Squared (R2): {r2:.2f}")

# Sample comparison
comparison = pd.DataFrame({
    'Actual_Price': y_test.values[:10],
    'Predicted_Price': predictions[:10].astype(int),
    'Age': X_test['age'].values[:10],
    'Make': X_test['make'].values[:10],
    'Model': X_test['model'].values[:10],
    'Cylinders': X_test['cylinders'].values[:10],
    'Trim': X_test['trim'].values[:10],
    'Drive': X_test['drive'].values[:10],
    'Fuel': X_test['fuel'].values[:10]
})
print("\nSample Predictions vs Actuals:")
print(comparison)

Model Performance:
Mean Absolute Error (MAE): $1,987.17
R-Squared (R2): 0.85

Sample Predictions vs Actuals:
   Actual_Price  Predicted_Price  Age        Make      Model    Cylinders  \
0          7200             9057   13       honda      civic  unspecified   
1          8500             8669    9         kia      forte  4 cylinders   
2          2700             3490   14      nissan      versa  4 cylinders   
3         10250            15244   15      toyota     tacoma  unspecified   
4         43000            43826    5       lexus         is  6 cylinders   
5         58000            54266    6       dodge        ram  unspecified   
6         12500            10573    7        jeep  gladiator  unspecified   
7         14000            15507    5  volkswagen      atlas  unspecified   
8         10000             9357   21       lexus      is300  unspecified   
9          4200             5209   27    cadillac    deville  8 cylinders   

          Trim        Drive    Fuel  
0    

In [75]:
# Save artifacts
os.makedirs('../api', exist_ok=True)
joblib.dump(model, '../api/model.pkl')
joblib.dump(ohe, '../api/ohe.pkl')
joblib.dump(X_train_final.columns.tolist(), '../api/model_columns.pkl')
print("\nArtifacts saved successfully!")


Artifacts saved successfully!
